In [ ]:
import re
from bs4 import BeautifulSoup
import requests
import pandas as pd
import json
from google.colab import files
import xml.etree.ElementTree as ET
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import io
import os
import time
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import warnings
import ast
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string
from itertools import chain
from collections import Counter
from operator import itemgetter
import time
import html
import unicodedata
import requests
from tqdm.auto import tqdm
from rapidfuzz import process, fuzz
# Download NLTK resources (if not already downloaded)
nltk.download('punkt')
nltk.download('stopwords')
warnings.simplefilter(action='ignore', category=FutureWarning)


Mounted at /content/drive


# **Nature_Extract_Author_Contribution**

In [ ]:
def get_soup(u):
  """
  Get The HTMl with Beautiful Soup
  """

  s=requests.get(u).content.decode('utf-8')
  html=s
  soup=BeautifulSoup(html, 'html.parser')
  return soup

def get_highest_page_count(soup):
  """
  Get The Highest Number Page In The Website. In bottom of the pages there are buttons to move on to the next page.
  Thus function return the biggest page number.
  """

  pagination_links = soup.find_all('a',attrs={'class':'c-pagination__link'})
  pattern_pagination=r'page=(\d+)'
  max_page=max([int(re.findall(pattern_pagination,str(pagination_links[i]))[0]) for i in range(len(pagination_links))])
  return max_page

def extract_title_urls_contribution(topics,type_url):
  """
  By each topic and his url I find the titles and articles' urls in each page.
  After that, to each article's url, I find:
  1.year publication of the article.
  2. The contibution of the authors. The contribuion came under the Section: "Author Information" under the title "Contribution".
  3. Author Address. The university and the country that the authors came from.
  4.Authors' Names

  The function downloads a dictionary by this structure: {topic:
                                            {Page Number:
                                            {title:..,url:..,contribution:..,year:..,address:..,authors:..}}}
  """

  for topic in topics:
    dict_all={}
    if type_url=='Nature Communication':
       u = "https://www.nature.com/subjects/"+topic+"/ncomms?searchType=journalSearch&sort=PubDate&page="
    else:
      #Nature
      u = "https://www.nature.com/subjects/"+topic+"/srep?searchType=journalSearch&sort=PubDate&page="

    soup=get_soup(u+str(1))
    try:
      max_page=get_highest_page_count(soup)
    except:
      max_page=1

    for i in range(1,max_page+1):
      dict1={'contribution':[],'url':[],'title':[],'authors':[],"address":[],'year':[]}
      print(i)
      if i>1:
        soup=get_soup(u+str(i))
      l = soup.find_all('a', href=lambda href: href and href.startswith("/articles/"))
      #all the urls shows in this pattern
      pattern_url = r'href="(.*?)"\s*itemprop="(.*?)"'
      urls=[re.findall(pattern_url, str(l[i]))[0][0] for i in range(len(l))]
      titles=[t.text.strip() for t in l]

      for j,url in enumerate(urls):
        title=titles[j]
        url_article="https://www.nature.com/"+topic+url
        soup_article=get_soup(url_article)
        year=soup_article.find('span', {'data-test': 'article-publication-year'}).text.strip()

        try:
          #contribution is under the class c-article__sub-heading under h3 tag and shows after <p .....</p>
          target_h3 = soup_article.find('h3', class_='c-article__sub-heading', text='Contributions').find_next('p')
          authour_contribution=target_h3.text.strip()
        except:
          authour_contribution=''
        try:
          address_elements = soup_article.find_all('p', class_='c-article-author-affiliation__address')
          address=[add.text.strip() for add in address_elements]
        except:
          authors=[]
          address=[]

        dict1['contribution']+=[authour_contribution]
        dict1['url']+=[url]
        dict1['title']+=[title]
        dict1['authors']+=[authors]
        dict1['address']+=[address]
        dict1['year']+=[year]
      dict_all[i]=dict1
      #after 30 pages download the file. Because the running is over 20 hours, so I divided to batches to save the results
      if i%30==0:
         with open(topic+".json", 'w') as json_file:
            json.dump(dict_all, json_file, indent=4)
         files.download(topic+".json")

    with open(topic+".json", 'w') as json_file:
      json.dump(dict_all, json_file, indent=4)
    files.download(topic+".json")

In [ ]:
##add the topic Names in the list topics.
topics=['ecology']
extract_title_urls_contribution(topics,'Nature Communication')

## Authors Extraction

In [ ]:
ROOT_FOLDERS = [
    #Path("/content/drive/MyDrive/data fro data_mining_project/nature"),
    Path("/content/drive/MyDrive/data fro data_mining_project/nature communication"),
]

BASE_URL = "https://www.nature.com"

OVERWRITE_ORIGINAL = False

REQUEST_DELAY = 0.5


session = requests.Session()

retry_strategy = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/149.0.0.0 Safari/537.36"
    )
})


def build_full_url(url):

    url = url.strip()

    if not url:
        raise ValueError("Url Empty")

    if url.startswith("http://") or url.startswith("https://"):
        return url

    if not url.startswith("/"):
        url = "/" + url

    return BASE_URL + url


def extract_authors_from_url(url):
    """
    In the datalayer has a dictionary
    מקבל URL של מאמר ומחזיר את רשימת המחברים מתוך dataLayer.
    """

    full_url = build_full_url(url)

    response = session.get(
        full_url,
        timeout=60,
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    script = soup.find(
        "script",
        attrs={"data-test": "dataLayer"},
    )

    if script is None:
        raise ValueError(
            f"not found a script with data-test='dataLayer': {full_url}"
        )

    script_text = script.get_text()

    match = re.search(
        r"window\.dataLayer\s*=\s*(\[.*?\])\s*;",
        script_text,
        flags=re.DOTALL,
    )

    if match is None:
        raise ValueError(
            f"not found window.dataLayer: {full_url}"
        )

    try:
        data_layer = json.loads(match.group(1))
    except json.JSONDecodeError as error:
        raise ValueError(
            f" error JSON dataLayer: {full_url}: {error}"
        ) from error

    for item in data_layer:
        if not isinstance(item, dict):
            continue

        content = item.get("content", {})

        if not isinstance(content, dict):
            continue

        content_info = content.get("contentInfo", {})

        if not isinstance(content_info, dict):
            continue

        authors = content_info.get("authors")

        if isinstance(authors, list):
            return authors

    raise ValueError(
        f"No authors in dataLayer: {full_url}"
    )


def get_output_path(input_path):
    """
    מחזיר את נתיב השמירה של הקובץ המעודכן.
    """

    if OVERWRITE_ORIGINAL:
        return input_path

    return input_path.with_name(
        f"{input_path.stem}_updated{input_path.suffix}"
    )


def get_error_path(input_path):
    """
    מחזיר נתיב לקובץ השגיאות של קובץ JSON מסוים.
    """

    return input_path.with_name(
        f"{input_path.stem}_authors_errors.json"
    )


def save_json(data, output_path):
    """
    שומר JSON בצורה מסודרת.
    """

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=4,
        )


def count_articles(data):
    """

    """

    total = 0

    if not isinstance(data, dict):
        return total

    for page_data in data.values():
        if not isinstance(page_data, dict):
            continue

        urls = page_data.get("url", [])

        if isinstance(urls, list):
            total += len(urls)

    return total


def process_json_file(input_path):
    """
    מעבד קובץ JSON יחיד ומעדכן בו את authors.
    """

    output_path = get_output_path(input_path)
    error_path = get_error_path(input_path)

    print("\n" + "=" * 100)
    print(f"Start work on the file:")
    print(input_path)
    print("=" * 100)

    try:
        with open(input_path, "r", encoding="utf-8") as file:
            data = json.load(file)
    except Exception as error:
        print(f"Can't read the file {error}")

        return {
            "file": str(input_path),
            "status": "file_read_error",
            "error": str(error),
        }

    if not isinstance(data, dict):
        print("No dictionary")

        return {
            "file": str(input_path),
            "status": "invalid_json_structure",
        }

    total_articles = count_articles(data)

    print(f"Total Articles in the file: {total_articles}")

    current_article = 0
    successful_articles = 0
    failed_urls = []

    for page_key, page_data in data.items():

        if not isinstance(page_data, dict):
            continue

        urls = page_data.get("url", [])
        titles = page_data.get("title", [])

        if not isinstance(urls, list):
            print(f" {page_key}")
            continue

        if not isinstance(titles, list):
            titles = []

        authors = []

        print()
        print(
            f"Page {page_key}: "
            f"{len(urls)} Articles"
        )

        for article_index, url in enumerate(urls):
            current_article += 1

            title = ""

            if article_index < len(titles):
                title = titles[article_index]

            print(
                f"\n[{current_article}/{total_articles}] "
                f"Page {page_key}, "
                f"Article {article_index + 1}/{len(urls)}"
            )

            print(f"URL: {url}")

            if title:
                print(f"Title: {title}")

            try:
                article_authors = extract_authors_from_url(url)

                authors.append(article_authors)
                successful_articles += 1

                print(
                    f"Number of articles authors Found: {len(article_authors)}"
                )

            except Exception as error:
                authors.append([])

                failed_urls.append({
                    "group": page_key,
                    "article_index": article_index,
                    "url": url,
                    "title": title,
                    "error": str(error),
                })

                print(f"שגיאה: {error}")

            time.sleep(REQUEST_DELAY)

        page_data["authors"] = authors

        save_json(data, output_path)

    if failed_urls:
        save_json(failed_urls, error_path)

        print(
            f"\n The errors in the file: \n{error_path}"
        )
    elif error_path.exists():
        error_path.unlink()

    print()
    print(f"The updated file saved in:")
    print(output_path)
    print(f"Success: {successful_articles}")
    print(f"Failures: {len(failed_urls)}")

    return {
        "file": str(input_path),
        "output_file": str(output_path),
        "total_articles": total_articles,
        "successful_articles": successful_articles,
        "failed_articles": len(failed_urls),
        "status": "completed",
    }


def find_json_files():
    """
    מוצא את כל קובצי ה-JSON בכל התיקיות, כולל תתי-תיקיות.
    """

    json_files = []

    for root_folder in ROOT_FOLDERS:

        if not root_folder.exists():
            print(f"The folder doesn't exist {root_folder}")
            continue

        for json_path in root_folder.rglob("*.json"):

            filename = json_path.name.lower()

            if filename.endswith("_updated.json"):
                continue

            if filename.endswith("_authors_errors.json"):
                continue

            json_files.append(json_path)

    return sorted(set(json_files))


json_files = find_json_files()

print(f"Json files found: {len(json_files)}")

all_results = []

for file_index, json_file in enumerate(json_files, start=1):

    print("\n" + "#" * 100)
    print(f"file {file_index}/{len(json_files)}")
    print("#" * 100)

    result = process_json_file(json_file)
    all_results.append(result)


summary_path = Path(
    "/content/drive/MyDrive/data fro data_mining_project/"
    "authors_extraction_summary.json"
)

save_json(all_results, summary_path)

total_articles = sum(
    result.get("total_articles", 0)
    for result in all_results
)

total_successful = sum(
    result.get("successful_articles", 0)
    for result in all_results
)

total_failed = sum(
    result.get("failed_articles", 0)
    for result in all_results
)


print(f"Files Number: {len(json_files)}")
print(f"Articles Number {total_articles}")
print(f"Successful Extraction Number: {total_successful}")
print(f"Failure Extraction Number: {total_failed}")


Streaming output truncated to the last 5000 lines.

Page 29: 50 Articles

[1401/1577] Page 29, Article 1/50
URL: /articles/ncomms6777
Title: The switching role of β-adrenergic receptor signalling in cell survival or death decision of cardiomyocytes
Number of articles authors Found: 7

[1402/1577] Page 29, Article 2/50
URL: /articles/ncomms6835
Title: Somatic mutations in arachidonic acid metabolism pathway genes enhance oral cancer post-treatment disease-free survival
Number of articles authors Found: 5

[1403/1577] Page 29, Article 3/50
URL: /articles/ncomms6778
Title: Serotonergic neurons respond to nutrients and regulate the timing of steroid hormone biosynthesis in Drosophila
Number of articles authors Found: 2

[1404/1577] Page 29, Article 4/50
URL: /articles/ncomms6755
Title: The long-chain alkane metabolism network of Alcanivorax dieselolei
Number of articles authors Found: 2

[1405/1577] Page 29, Article 5/50
URL: /articles/ncomms6732
Title: TRPA1 is essential for the vascular 

# **Open Corpus**

In [ ]:
zip_file_path = '/content/drive/MyDrive/data_mining_project/allofplos.zip'

# Specify the path where you want to extract the contents
extracted_folder_path = '/content/extracted_folder/'

# Open the zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    # Extract all contents to the specified folder
    zip_ref.extractall(extracted_folder_path)

In [ ]:
def get_title(root):
  '''
  Each file is XML file. the title tag in the xml file is .//title-group.
  The function return full title name
  '''
  title_group = root.find('.//title-group')
  article_title = title_group.find('.//article-title')
  title = ET.tostring(article_title, encoding='utf-8').decode('utf-8').strip()
  title = re.sub(r'<.*?>', '', title)
  return title

def get_year(root):
  '''
  Each file is XML file. the date tag in the xml file is .///pub-date. From the date tag, I extract the year
  The function returns the publication year of the article.
  '''
  date=root.find('.//pub-date')
  year=date.find('.//year')
  return year.text

#type1:
def get_contribution(root):
    '''
    Each file is XML file. the contribution tag in the xml file is ..//*[@fn-type='con'. From the contribution tag,
    I extract the full text of the contribution
    The function returns contribution types for each author.
    '''
    try:
      contribution=ET.tostring(root.findall(".//*[@fn-type='con']")[0], encoding='utf-8', method='text').decode('utf-8').strip()
    except:
      contribution=[]

    return contribution

def get_address(root):
  address=root.findall('.//addr-line')
  aff=root.findall('.//aff')
  addr_line_texts={}
  if len(address)<=2:
    try:
      addr_line_texts = {'1': ET.tostring(address[0], encoding='utf-8', method='text').decode('utf-8').strip()}
    except:
      return {}
  else:
    for i,elem in enumerate(address):
      try:
        aff_num=ET.tostring(aff[i][0], encoding='utf-8', method='text').decode('utf-8')[0]
      except:
        aff_num=ET.tostring(aff[i], encoding='utf-8', method='text').decode('utf-8')[0]
      if aff_num.isdigit()==True:
        addr_line_texts[aff_num]=ET.tostring(elem, encoding='utf-8', method='text').decode('utf-8').strip()

  return addr_line_texts

def get_names_address(root,contrib_found):
  type_xml=1
  lst_names_affs=[]
  contrib_grpoup_name=root.findall(".//*[@contrib-type='author']")
  if contrib_grpoup_name==[]:
    return lst_names_affs,0
  dict_names_address_contrib={'full name':[],'address':[],'roles':[]}
  if contrib_grpoup_name[0].findall('.//role')==[]:
    type_xml=0
    dict_names_address_contrib={'full name':[],'address':[]}
    if contrib_found==[]:
      return lst_names_affs,type_xml

  for i,elem in enumerate(contrib_grpoup_name):
    affs=[]
    dict_names_address_contrib2={}
    roles_to_each_author=[]
    first_name=elem.findall('.//surname')
    last_names=elem.findall('.//given-names')
    if first_name==[]:
      continue
    aff=elem.findall('.//sup')
    if type_xml==1:
      roles= elem.findall('.//role')
      for role in roles:
        roles_to_each_author.append(ET.tostring(role, encoding='utf-8', method='text').decode('utf-8').strip())

    addr_line_texts=get_address(root)
    if len(aff)>=1:
      for j , aff_id in enumerate(aff):
        try:
          aff_num=ET.tostring(aff_id, encoding='utf-8', method='text').decode('utf-8').strip()
          if aff_num=='*':
            affs.append(list(addr_line_texts.values()))
          else:
            affs.append(addr_line_texts[aff_num])
        except:
          continue

    elif aff==[]:
      if addr_line_texts!={}:
        if '1' in addr_line_texts.keys():
          affs.append(addr_line_texts['1'])

    else:
        try:
          affs.append(addr_line_texts[ET.tostring(aff[0], encoding='utf-8', method='text').decode('utf-8').strip()])
        except:
          continue
    if last_names==[]:
      full_name=ET.tostring(first_name[0], encoding='utf-8', method='text').decode('utf-8').strip()
    else:
      full_name=ET.tostring(first_name[0], encoding='utf-8', method='text').decode('utf-8').strip()+' '+ET.tostring(last_names[0], encoding='utf-8', method='text').decode('utf-8').strip()

    dict_names_address_contrib2['full name']=full_name
    dict_names_address_contrib2['address']=affs
    if 'roles' in dict_names_address_contrib.keys():
      dict_names_address_contrib2['roles']=roles_to_each_author
    lst_names_affs.append(dict_names_address_contrib2)
  return lst_names_affs,type_xml

### **Create Json With Relevant Features:**

In [ ]:
path='/content/extracted_folder/'
files = os.listdir(path)
dict_files_meta_data={}
for i in range(len(files)):
  file_name = os.path.join(path, files[i])
  with open(file_name, 'r') as file:
        content = file.read()

  try:
    root = ET.fromstring(content)
  except:
    continue
  # Now 'root' is an ElementTree object that you can work with
  root = root.find('.//article-meta')
  contribution=get_contribution(root)
  dict_name_address_role=get_names_address(root,contribution)
  if (contribution==[] and dict_name_address_role[1]==0) or (contribution==[] and dict_name_address_role[0]==[]) :
    #there is nor contribution in this article
    continue
  if contribution!=[] and dict_name_address_role[0]==[]:
    dict_file={'title':title,'year':year,'contribution':contribution}
    dict_files_meta_data[file_name]=dict_file
    continue

  year=get_year(root)
  title=get_title(root)
  if 'roles' in dict_name_address_role[0][0].keys():
    dict_file={'title':title,'year':year,'name_address_contribution':dict_name_address_role[0]}
  else:
    dict_file={'title':title,'year':year,'name_address':dict_name_address_role[0],'contribution':contribution}

  dict_files_meta_data[file_name]=dict_file

with open("plosone.json", 'w') as json_file:
    json.dump(dict_files_meta_data, json_file, indent=4)


In [ ]:
file_name = os.path.join(path, files[43039])
with open(file_name, 'r') as file:
    content = file.read()


In [ ]:
print(content)

# Extract Contribution DF & Roles DF:

In [ ]:
def create_df_contribution(file_names,path):

  df_roles = pd.DataFrame(columns=['title', 'year', 'contribtuion type','address','full name'])
  df_contribution=pd.DataFrame(columns=['title', 'year', 'contribtuion type','authors','address','full name'])

  for file_name in file_names:
    print(file_name)
    path_file=path+file_name
    with open(path_file, 'r') as file:
        data = json.load(file)
    for i,key in enumerate(list(data.keys())):

      # First type of representaion of contribution types. It's a list of roles (Contribution types) for each author.
      #For exmple: roles: name_address_contribution:{full name: [Brand James, Li Xiu]
      #                                             roles:[analyze the data, wrtie the paper, performed the experients],
      #                                                   [data collection, visualization]...
      if 'name_address_contribution' in data[key]:
        lst_full_names,lst_address,lst_roles=[],[],[]
        title=data[key]['title']
        year=data[key]['year']
        try:
            for dict_names_Address in data[key]['name_address_contribution']:
              lst_full_names.append(dict_names_Address['full name'])
              lst_address.append(dict_names_Address['address'])
              lst_roles.append(dict_names_Address['roles'])
        except:
            continue

        new_row = {'title': title, 'year': year, 'contribtuion type': lst_roles,
                    'address':lst_address,'full name':lst_full_names}
        df_roles = df_roles.append(new_row, ignore_index=True)


      # Second type of representaion of contribution types. It's a string that in the string there are the roles and authors.
      #For exmple: Analyze the data: MM N.Raul PJ. Write the paper: N. Moli MN KK. ...
      if 'contribution' in data[key]:
        title=data[key]['title']
        year=data[key]['year']
        lst_full_names,lst_address=[],[]

        try:
          for dict_names_Address in data[key]['name_address']:
            lst_full_names.append(dict_names_Address['full name'])
            lst_address.append(dict_names_Address['address'])
        except:
          lst_full_names,lst_address=[],[]

        try:
          first_contrib=' '.join(data[key]['contribution'].split("\n")).split(":")

          #somethimes the first "contribution type" is a declaration. As a r result if it happened we will take the second match as
          # the first contribution.
          #For example: The author(s) have made the following declarations about their contributions: Analyze the data: KN PP ....
          if "The author(s) have made the following declarations about their contributions" in first_contrib:
            first_contrib=[first_contrib[1]]
          else:
            first_contrib=[first_contrib[0]]
        except:
          contribution_types=[]

        # I wanted to get the contibution types. They are in same stracture: ./: , space and capital letter. After that could be
        # spaces, letters (small letters) and pancuation like: -,/&.
        #As a result: (?<=\. ): Positive lookbehind assertion ensuring the match is preceded by a period and a space.
        #([A-Z][^.:]+): Captures a sequence starting with an uppercase letter, followed by one or more characters except colon and period. This covers the names of roles.
        #(?=:): Positive lookahead assertion ensuring the match is followed by a colon.
        pattern = r"(?<=\. )([A-Z][^.:]+(?=:))"
        other_contrib_types = re.findall(pattern, data[key]['contribution'])
        contribution_types=contribution_types+other_contrib_types

        #Get all the authorts that between contribution types/ from contribution type to end of the sentence.
        # From the example, I want to take: [MM N.Raul PJ. , N. Moli MN KK]
        pattern2 = '|'.join(map(re.escape, contribution_types))
        author_foreach_contrib_type = re.split(pattern2, data[key]['contribution'])

        new_row = {'title': title, 'year': year, 'contribtuion type': contribution_types,'authors':author_foreach_contrib_type,
                  'address':lst_address,'full name':lst_full_names}
        df_contribution = df_contribution.append(new_row, ignore_index=True)

  return df_contribution,df_roles

In [ ]:
path="/content/drive/MyDrive/data fro data_mining_project/allplos/"
file_names = os.listdir(path)
df_contribution,roles_df=create_df_contribution(file_names,path)

### DF Contribution & Roles- One plus:

In [ ]:
# in each cell in contribute type the value is: '['conceived','visualization'...]'.
df_contribution['contribtuion type'] = df_contribution['contribtuion type'].apply(ast.literal_eval)
df_contribution['authors']=df_contribution['authors'].apply(ast.literal_eval)
df_contribution['full name']=df_contribution['full name'].apply(ast.literal_eval)
df_contribution['address']=df_contribution['address'].apply(ast.literal_eval)

roles_df['contribtuion type'] = roles_df['contribtuion type'].apply(ast.literal_eval)
roles_df['full name'] = roles_df['full name'].apply(ast.literal_eval)
roles_df['address']=roles_df['address'].apply(ast.literal_eval)
roles_df['contribtuion type_correction']=roles_df['contribtuion type_correction'].apply(ast.literal_eval)


In [ ]:
def remove_space_from_list(lst):
    return [item for item in lst if item != '']

def remove_colon_and_dot(lst):
      return [item.replace(':', '')for item in lst]

df_contribution['authors'] = df_contribution['authors'].apply(remove_space_from_list)
df_contribution['authors'] = df_contribution['authors'].apply(remove_colon_and_dot)

In [ ]:
def Uniform_DF_Contrib_Type(lst):
  '''
  The function recieves a list of contribution of an author. The goal of this function is to create uniform names
  for a type of a contribution.
  '''
  full_contrib_types_article=[]
  for i,sub in enumerate(lst):
    full_contrib_types_author=[]
    for elem in sub:
      lst_new_contrib_Type=[]
      tokens = word_tokenize(elem)
      for token in tokens:
        if token not in '"#$%&()*+, -./:;<=>?@[\]^_`{|}~:-&':
          token=token.lower()
          lst_new_contrib_Type.append(token)
      contrib_type=' '.join(lst_new_contrib_Type)

      #there are 2 types of contribution that different each other. Writing the draft the article and
      # writing and reviewing the submitted article (not the draft)
      if 'original draft' in contrib_type:
        full_contrib_types_author.append('writing – original draft')
      elif 'review' in contrib_type:
        full_contrib_types_author.append('writing – review editing')
      #elif contrib_type in list(contrib_value_dict.keys()):
      else:
        full_contrib_types_author.append(contrib_type)

    full_contrib_types_article.append(full_contrib_types_author)
  return full_contrib_types_article

roles_df['contribtuion type_correction'] = roles_df['contribtuion type'].apply(Uniform_DF_Contrib_Type)

In [ ]:
roles_df.to_csv('roles_df_correction.csv')
df_contribution.to_csv('contribution_df.csv')

### Assign Domain for each DF:

In [ ]:
API_KEY = "YOUR_OPENALEX_API_KEY"

TITLE_COL = "title"
YEAR_COL = "year"

DOI_PREFIX = "10.1371"
FUZZY_CUTOFF = 95

OPENALEX_CACHE = "openalex_plos_works_topics.parquet"

FILES = {
    "contribution": {
        "input": "/content/drive/MyDrive/data_mining_project/contribution_df.csv",
        "output_csv": "df_with_openalex_fields.csv",
        "output_parquet": "df_with_openalex_fields.parquet",
    },
    "roles": {
        "input": "/content/drive/MyDrive/data_mining_project/roles_df_correction.csv",
        "output_csv": "roles_df_with_fields.csv",
        "output_parquet": "roles_df_with_fields.parquet",
    },
}

def normalize_title(s):
    """
    Normalize an article title for reliable matching by removing HTML,
    accents, punctuation, capitalization, and unnecessary whitespace.
    """
    if pd.isna(s):
        return ""

    s = html.unescape(str(s))
    s = unicodedata.normalize("NFKD", s)
    s = "".join(
        ch for ch in s
        if not unicodedata.combining(ch)
    )
    s = s.lower()

    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def safe_get(d, *keys):
    """
    Safely retrieve a value from a nested dictionary.
    Return None if one of the requested keys is missing.
    """
    current = d

    for key in keys:
        if not isinstance(current, dict):
            return None

        current = current.get(key)

    return current


def parse_openalex_work(work):
    """
    Extract the article identifiers and classification hierarchy
    from a single OpenAlex work record.
    """
    primary_topic = work.get("primary_topic") or {}

    return {
        "openalex_id": work.get("id"),
        "doi": work.get("doi"),
        "oa_title": work.get("display_name"),
        "oa_year": work.get("publication_year"),

        "topic_id": primary_topic.get("id"),
        "topic": primary_topic.get("display_name"),

        "subfield_id": safe_get(
            primary_topic, "subfield", "id"
        ),
        "subfield": safe_get(
            primary_topic, "subfield", "display_name"
        ),

        "field_id": safe_get(
            primary_topic, "field", "id"
        ),
        "field": safe_get(
            primary_topic, "field", "display_name"
        ),

        "domain_id": safe_get(
            primary_topic, "domain", "id"
        ),
        "domain": safe_get(
            primary_topic, "domain", "display_name"
        ),

        # Used internally only for selecting duplicate records.
        "_topic_score": primary_topic.get("score"),
    }


def request_openalex(params, max_retries=8):
    """
    Send a request to the OpenAlex Works API.
    Retry temporary rate-limit and server errors automatically.
    """
    url = "https://api.openalex.org/works"

    for attempt in range(max_retries):
        response = requests.get(
            url,
            params=params,
            timeout=90
        )

        if response.status_code == 200:
            return response.json()

        if response.status_code in {
            429, 500, 502, 503, 504
        }:
            sleep_time = min(60, 2 ** attempt)

            print(
                f"OpenAlex status {response.status_code}. "
                f"Sleeping {sleep_time}s..."
            )

            time.sleep(sleep_time)
            continue

        raise RuntimeError(
            f"OpenAlex error {response.status_code}: "
            f"{response.text[:1000]}"
        )

    raise RuntimeError("Too many OpenAlex retries")


def prepare_openalex(oa):
    """
    Normalize the OpenAlex cache, remove invalid records,
    and keep one record for every normalized title and year.
    """
    oa = oa.copy()

    oa["oa_year"] = pd.to_numeric(
        oa["oa_year"],
        errors="coerce"
    ).astype("Int64")

    oa["_topic_score"] = pd.to_numeric(
        oa.get("_topic_score"),
        errors="coerce"
    )

    oa["title_norm"] = oa["oa_title"].map(
        normalize_title
    )

    oa = oa[
        oa["title_norm"].ne("") &
        oa["oa_year"].notna()
    ].copy()

    oa = (
        oa.sort_values(
            ["title_norm", "oa_year", "_topic_score"],
            ascending=[True, True, False],
            na_position="last"
        )
        .drop_duplicates(
            ["title_norm", "oa_year"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    return oa


def fetch_plos_works_by_years(
    years,
    api_key,
    doi_prefix="10.1371"
):
    """
    Download all PLOS works from OpenAlex for the requested years
    and return a cleaned DataFrame containing their classifications.
    """
    all_rows = []

    years = sorted({
        int(year)
        for year in years
        if pd.notna(year)
    })

    for year in tqdm(
        years,
        desc="Downloading PLOS works from OpenAlex"
    ):
        cursor = "*"
        year_rows = 0

        while True:
            params = {
                "api_key": api_key,
                "filter": (
                    f"doi_starts_with:{doi_prefix},"
                    f"publication_year:{year}"
                ),
                "select": (
                    "id,doi,display_name,"
                    "publication_year,primary_topic"
                ),
                "per_page": 100,
                "cursor": cursor,
            }

            data = request_openalex(params)
            results = data.get("results", [])

            if not results:
                break

            for work in results:
                all_rows.append(
                    parse_openalex_work(work)
                )

            year_rows += len(results)

            cursor = data.get(
                "meta", {}
            ).get("next_cursor")

            if not cursor:
                break

        print(
            f"Year {year}: downloaded "
            f"{year_rows:,} works"
        )

    oa = pd.DataFrame(all_rows)

    if oa.empty:
        raise ValueError(
            "No OpenAlex records were downloaded."
        )

    return prepare_openalex(oa)


def enrich_dataframe(
    df,
    oa,
    dataset_name,
    output_csv,
    output_parquet
):
    """
    Match one input DataFrame with OpenAlex using exact title-year
    matching first and fuzzy matching for the remaining articles.
    """
    df_work = df.copy()

    # Remove fields from a previous run.
    old_columns = [
        "title_norm",
        "openalex_id",
        "doi",
        "oa_title",
        "oa_year",
        "topic_id",
        "topic",
        "topic_score",
        "_topic_score",
        "subfield_id",
        "subfield",
        "field_id",
        "field",
        "domain_id",
        "domain",
        "match_type",
        "match_score",
    ]

    df_work = df_work.drop(
        columns=[
            column for column in old_columns
            if column in df_work.columns
        ],
        errors="ignore"
    )

    df_work[TITLE_COL] = (
        df_work[TITLE_COL].astype(str)
    )

    df_work[YEAR_COL] = pd.to_numeric(
        df_work[YEAR_COL],
        errors="coerce"
    ).astype("Int64")

    df_work["title_norm"] = (
        df_work[TITLE_COL].map(normalize_title)
    )

    years = sorted(
        df_work[YEAR_COL]
        .dropna()
        .astype(int)
        .unique()
    )

    print()
    print("=" * 70)
    print(f"PROCESSING: {dataset_name}")
    print("=" * 70)
    print("Input rows:", len(df_work))
    print(
        "Unique normalized titles:",
        df_work["title_norm"].nunique()
    )

    if years:
        print(
            "Years:",
            min(years),
            "-",
            max(years),
            "| n_years:",
            len(years)
        )

    topic_columns = [
        "openalex_id",
        "doi",
        "oa_title",
        "oa_year",
        "topic_id",
        "topic",
        "subfield_id",
        "subfield",
        "field_id",
        "field",
        "domain_id",
        "domain",
        "title_norm",
    ]

    matched = df_work.merge(
        oa[topic_columns],
        left_on=["title_norm", YEAR_COL],
        right_on=["title_norm", "oa_year"],
        how="left",
        suffixes=("", "_oa")
    )

    matched["match_type"] = None
    matched["match_score"] = None

    exact_mask = matched["openalex_id"].notna()

    matched.loc[
        exact_mask,
        "match_type"
    ] = "exact"

    matched.loc[
        exact_mask,
        "match_score"
    ] = 100

    print()
    print(
        "Exact matched:",
        exact_mask.sum(),
        "/",
        len(matched)
    )

    print(
        "Exact unmatched:",
        matched["openalex_id"].isna().sum()
    )

    oa_by_year = {
        int(year): subset.reset_index(drop=True)
        for year, subset in (
            oa.dropna(subset=["oa_year"])
            .groupby("oa_year")
        )
    }

    unmatched_indexes = matched.index[
        matched["openalex_id"].isna()
    ].tolist()

    print()
    print(
        "Need fuzzy matching:",
        len(unmatched_indexes)
    )

    columns_to_fill = [
        "openalex_id",
        "doi",
        "oa_title",
        "oa_year",
        "topic_id",
        "topic",
        "subfield_id",
        "subfield",
        "field_id",
        "field",
        "domain_id",
        "domain",
    ]

    fuzzy_count = 0

    for index in tqdm(
        unmatched_indexes,
        desc=f"Fuzzy matching {dataset_name}"
    ):
        row = matched.loc[index]

        if (
            not row["title_norm"] or
            pd.isna(row[YEAR_COL])
        ):
            continue

        candidates = oa_by_year.get(
            int(row[YEAR_COL])
        )

        if candidates is None or candidates.empty:
            continue

        result = process.extractOne(
            row["title_norm"],
            candidates["title_norm"].tolist(),
            scorer=fuzz.token_sort_ratio,
            score_cutoff=FUZZY_CUTOFF
        )

        if result is None:
            continue

        _, score, position = result
        hit = candidates.iloc[position]

        for column in columns_to_fill:
            matched.at[index, column] = hit.get(
                column
            )

        matched.at[
            index,
            "match_type"
        ] = "fuzzy"

        matched.at[
            index,
            "match_score"
        ] = score

        fuzzy_count += 1

    print("Fuzzy matched:", fuzzy_count)

    total = len(matched)
    matched_count = (
        matched["openalex_id"].notna().sum()
    )
    unmatched_count = (
        matched["openalex_id"].isna().sum()
    )

    print()
    print("=" * 60)
    print("FINAL MATCH REPORT")
    print("=" * 60)
    print(f"Total input rows: {total:,}")
    print(
        f"Matched total:    {matched_count:,} "
        f"({matched_count / total:.2%})"
    )
    print(
        f"Unmatched:        {unmatched_count:,} "
        f"({unmatched_count / total:.2%})"
    )

    print()
    print("Match type counts:")
    print(
        matched["match_type"]
        .value_counts(dropna=False)
    )

    print()
    print(
        "Missing domain:",
        matched["domain"].isna().sum()
    )
    print(
        "Missing field:",
        matched["field"].isna().sum()
    )
    print(
        "Missing subfield:",
        matched["subfield"].isna().sum()
    )
    print(
        "Missing topic:",
        matched["topic"].isna().sum()
    )

    # Ensure topic_score does not appear in the output.
    matched = matched.drop(
        columns=["topic_score", "_topic_score"],
        errors="ignore"
    )

    matched.to_parquet(
        output_parquet,
        index=False
    )

    matched.to_csv(
        output_csv,
        index=False
    )

    print()
    print("Saved:")
    print(output_parquet)
    print(output_csv)

    df_topics = matched[
        [
            TITLE_COL,
            YEAR_COL,
            "domain",
            "field",
            "subfield",
            "topic",
            "openalex_id",
            "doi",
            "match_type",
            "match_score",
            "oa_title",
        ]
    ].copy()

    print()
    print("Preview:")
    display(df_topics.head())

    print()
    print("Domain counts:")
    display(
        matched["domain"]
        .value_counts(dropna=False)
        .rename_axis("domain")
        .reset_index(name="count")
    )

    print()
    print("Field counts:")
    display(
        matched["field"]
        .value_counts(dropna=False)
        .rename_axis("field")
        .reset_index(name="count")
    )

    unmatched_examples = matched[
        matched["openalex_id"].isna()
    ][
        [TITLE_COL, YEAR_COL, "title_norm"]
    ].head(30)

    print()
    print("Unmatched examples:")
    display(unmatched_examples)

    return matched

dataframes = {}

for name, settings in FILES.items():
    dataframes[name] = pd.read_csv(
        settings["input"]
    )

    print(
        f"Loaded {name}: "
        f"{dataframes[name].shape}"
    )

all_years = set()

for dataframe in dataframes.values():
    file_years = pd.to_numeric(
        dataframe[YEAR_COL],
        errors="coerce"
    ).dropna()

    all_years.update(
        file_years.astype(int).unique()
    )

print()
print(
    f"Required years: {min(all_years)}–{max(all_years)}"
)
print(
    f"Number of required years: {len(all_years)}"
)


if os.path.exists(OPENALEX_CACHE):
    oa = pd.read_parquet(
        OPENALEX_CACHE
    )

    # Support an older cache containing topic_score.
    if (
        "_topic_score" not in oa.columns and
        "topic_score" in oa.columns
    ):
        oa["_topic_score"] = oa["topic_score"]

    oa = prepare_openalex(oa)

    print(
        "Loaded OpenAlex cache:",
        oa.shape
    )
else:
    oa = pd.DataFrame()
    print("OpenAlex cache not found.")

cached_years = (
    set(
        oa["oa_year"]
        .dropna()
        .astype(int)
    )
    if not oa.empty
    else set()
)

missing_years = all_years - cached_years

print(
    "Years already in cache:",
    len(cached_years)
)
print(
    "Missing years:",
    sorted(missing_years)
)

if missing_years:
    downloaded = fetch_plos_works_by_years(
        years=missing_years,
        api_key=API_KEY,
        doi_prefix=DOI_PREFIX
    )

    oa = prepare_openalex(
        pd.concat(
            [oa, downloaded],
            ignore_index=True
        )
    )

    oa.to_parquet(
        OPENALEX_CACHE,
        index=False
    )

    print(
        "Updated OpenAlex cache:",
        oa.shape
    )

print()
print(
    "OpenAlex candidate works:",
    len(oa)
)

display(
    oa[
        [
            "oa_title",
            "oa_year",
            "domain",
            "field",
            "subfield",
            "topic",
        ]
    ].head()
)


results = {}

for name, settings in FILES.items():
    results[name] = enrich_dataframe(
        df=dataframes[name],
        oa=oa,
        dataset_name=name,
        output_csv=settings["output_csv"],
        output_parquet=settings["output_parquet"]
    )

roles_df rows: 131781
unique articles in roles_df: 91914
PLOS works rows: 386944
PLOS columns:
['openalex_id', 'doi', 'oa_title', 'oa_year', 'topic_id', 'topic', 'topic_score', 'subfield_id', 'subfield', 'field_id', 'field', 'domain_id', 'domain', 'title_norm']
Clean PLOS works: 386944
PLOS years: 2003 - 2023

Exact matched unique articles: 91130 / 91914
Exact unmatched unique articles: 784

Need fuzzy matching: 784


Fuzzy matching:   0%|          | 0/784 [00:00<?, ?it/s]

Fuzzy matched: 65

UNIQUE ARTICLE MATCH REPORT
Unique articles: 91,914
Matched:         91,195 (99.22%)
Unmatched:       719 (0.78%)

Match type counts:
match_type
exact    91130
None       719
fuzzy       65
Name: count, dtype: int64

FULL ROLES_DF REPORT
roles_df rows:             131,781
rows with OpenAlex match:  130,740
rows without match:        1,041

Rows missing domain: 1041
Rows missing field: 1041
Rows missing subfield: 1041
Rows missing topic: 1041

Saved:
article_matches_with_fields.csv
roles_df_with_fields.csv
article_matches_with_fields.parquet
roles_df_with_fields.parquet


,title,year,domain,field,subfield,topic,match_type,match_score,oa_title
0,"The ecology of immune state in a wild mammal, ...",2018,Physical Sciences,Environmental Science,Ecology,Wildlife Ecology and Conservation,exact,None,"The ecology of immune state in a wild mammal, ..."
1,The mechanism of complex formation between cal...,2021,Physical Sciences,Chemistry,Spectroscopy,Mass Spectrometry Techniques and Applications,exact,None,The mechanism of complex formation between cal...
2,Effect of firearms legislation on suicide and ...,2020,Social Sciences,Psychology,Clinical Psychology,Suicide and Self-Harm Studies,exact,None,Effect of firearms legislation on suicide and ...
3,A computational framework to assess genome-wid...,2019,Life Sciences,Agricultural and Biological Sciences,Plant Science,Chromosomal and Genetic Variations,exact,None,A computational framework to assess genome-wid...
4,Safety and immunogenicity following co-adminis...,2023,Health Sciences,Medicine,"Public Health, Environmental and Occupational ...",Mosquito-borne diseases and control,exact,None,Safety and immunogenicity following co-adminis...
5,Increase of CD4+CD25highFoxP3+ cells impairs i...,2021,Health Sciences,Medicine,Infectious Diseases,Tuberculosis Research and Epidemiology,exact,None,Increase of CD4+CD25highFoxP3+ cells impairs i...
6,Generalized structural equations improve sexua...,2017,Life Sciences,Agricultural and Biological Sciences,"Ecology, Evolution, Behavior and Systematics",Plant and animal studies,exact,None,Generalized structural equations improve sexua...
7,Approaches and geographical locations of respe...,2023,Health Sciences,Medicine,Obstetrics and Gynecology,Maternal and Perinatal Health Interventions,exact,None,Approaches and geographical locations of respe...
8,Effects of dexmedetomidine as a perineural adj...,2020,Health Sciences,Medicine,Surgery,Anesthesia and Pain Management,exact,None,Effects of dexmedetomidine as a perineural adj...
9,Cortical signatures of auditory object binding...,2022,Life Sciences,Neuroscience,Cognitive Neuroscience,Neural dynamics and brain function,exact,None,Cortical signatures of auditory object binding...


Domain counts:


,count,count
0,Health Sciences,40170
1,Life Sciences,27354
2,Physical Sciences,12692
3,Social Sciences,10979
4,NaN,719


Field counts:


,count,count
0,Medicine,35757
1,"Biochemistry, Genetics and Molecular Biology",13233
2,Agricultural and Biological Sciences,5830
3,Environmental Science,5228
4,Psychology,4203
5,Neuroscience,4143
6,Immunology and Microbiology,3893
7,Social Sciences,3810
8,Health Professions,2487
9,Engineering,2448


Unmatched examples:


,title,year,title_norm
111,LGBTQ+ identities in the Indian audiovisual ad...,2024,lgbtq identities in the indian audiovisual adv...
121,Chemobiosis reveals tardigrade tun formation i...,2024,chemobiosis reveals tardigrade tun formation i...
228,Psychometric properties and measurement invari...,2024,psychometric properties and measurement invari...
286,Hypertensive rats show increased renal excreti...,2024,hypertensive rats show increased renal excreti...
304,Hepatocyte ballooning and steatosis in early a...,2024,hepatocyte ballooning and steatosis in early a...
442,Educational cooperation in the perspective of ...,2024,educational cooperation in the perspective of ...
809,Neural speech restoration at the cocktail part...,2020,neural speech restoration at the cocktail part...
825,"""I told myself, be bold and go and test"": Moti...",2024,i told myself be bold and go and test motivato...
881,"Socioeconomic per-case costs of stroke, myocar...",2024,socioeconomic per case costs of stroke myocard...
1074,Trends of inequality in DPT3 immunization serv...,2024,trends of inequality in dpt3 immunization serv...
